In [2]:
!pip install langchain

In [3]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00


In [4]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 3.9 MB/s eta 0:00:00


#### **ConversationSummaryMemory**

In [9]:
import os
from google.colab import userdata



In [14]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [13]:
LANG_API_KEY = userdata.get("LANG_API_KEY")
os.environ["LANG_API_KEY"] = LANG_API_KEY

In [15]:
LANGSMITH_TRACING="true"
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=LANG_API_KEY
LANGSMITH_PROJECT="summery_memo"
GROQ_API_KEY=GROQ_API_KEY

In [16]:
# impor the model
from langchain_groq import ChatGroq
model = ChatGroq(model_name="Gemma2-9b-It")

In [17]:
# invoke the model
import warnings
warnings.filterwarnings('ignore')

In [18]:
model.invoke("Hi, What's up").content

"Not much, just hanging out in the digital world, waiting to be helpful! How about you? What's going on? 😊\n"

In [19]:
# import ConversationSummaryMemory
from langchain.memory import ConversationSummaryMemory
from langchain.memory import ChatMessageHistory

In [20]:
# instance of memory
memory = ConversationSummaryMemory(
    llm=model,return_message=True
)

###### Type-1: How we can store the memory

In [21]:
memory.save_context(
    {"input":"My name is Yogesh I live in Jungle"},
    {"output":"That's nice to meet you, How are you?"}

)

In [22]:
memory.load_memory_variables({})

{'history': 'New summary: \n\nA human named Yogesh introduces himself, saying he lives in Jungle. The AI responds politely, asking how Yogesh is doing. \n'}

In [ ]:
# i can make the variable of the summary and then access the content amoung itself
summary = memory.load_memory_variables({})
summary["history"]

In [23]:
# another context to the llm
memory.save_context(
    {"input": "It's a machine learning project focused on healthcare."},
    {"output": "Sounds exciting! Are they building a prediction model or something else"}
)
memory.save_context(
    {"input": "Yes, they’re working on a model to predict patient outcomes."},
    {"output": "Impressive! How far along are they with the project?"}
)

In [24]:
memory.load_memory_variables({})

{'history': "A human named Yogesh introduces himself, saying he lives in Jungle. The AI responds politely, asking how Yogesh is doing. Yogesh tells the AI he's working on a machine learning project focused on healthcare, specifically a model to predict patient outcomes.  The AI expresses excitement and asks how far along Yogesh is with the project. \n"}

In [25]:
print('Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.\n\nEXAMPLE\nCurrent summary:\nThe human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.\n\nNew lines of conversation:\nHuman: Why do you think artificial intelligence is a force for good?\nAI: Because artificial intelligence will help humans reach their full potential.\n\nNew summary:\nThe human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.\nEND OF EXAMPLE\n\nCurrent summary:\n{summary}\n\nNew lines of conversation:\n{new_lines}\n\nNew summary:')


Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:


In [26]:
# checking the main chat_memory
memory.chat_memory

InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Yogesh I live in Jungle', additional_kwargs={}, response_metadata={}), AIMessage(content="That's nice to meet you, How are you?", additional_kwargs={}, response_metadata={}), HumanMessage(content="It's a machine learning project focused on healthcare.", additional_kwargs={}, response_metadata={}), AIMessage(content='Sounds exciting! Are they building a prediction model or something else', additional_kwargs={}, response_metadata={}), HumanMessage(content='Yes, they’re working on a model to predict patient outcomes.', additional_kwargs={}, response_metadata={}), AIMessage(content='Impressive! How far along are they with the project?', additional_kwargs={}, response_metadata={})])

##### Type-2 Of getting summarymemory

In [27]:
# actual messages
messages = memory.chat_memory.messages

In [28]:
messages

[HumanMessage(content='My name is Yogesh I live in Jungle', additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's nice to meet you, How are you?", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="It's a machine learning project focused on healthcare.", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sounds exciting! Are they building a prediction model or something else', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Yes, they’re working on a model to predict patient outcomes.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Impressive! How far along are they with the project?', additional_kwargs={}, response_metadata={})]

In [29]:
previous_summary = ""

memory.predict_new_summary(messages,previous_summary)

'The human introduces themself as Yogesh and says they live in Jungle. The AI is polite and asks how Yogesh is doing.  Yogesh explains that they are part of a machine learning project focused on healthcare. The AI expresses excitement and asks if the project is building a prediction model. Yogesh confirms that they are working on a model to predict patient outcomes. The AI  compliments their work and asks how far along the project is. \n\n\n'

##### Type-3

In [30]:
history = ChatMessageHistory()

In [31]:
history

InMemoryChatMessageHistory(messages=[])

In [32]:
history.add_user_message('hi')
history.add_ai_message("What's up how can i guide you")

In [33]:
ConversationSummaryMemory.from_messages(
    llm=model,
    chat_memory=history,
    memory_key="summary",
    human_prefix="Human",
    ai_prefix="AI",

)

ConversationSummaryMemory(llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7b08026bd210>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7b0802447e90>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********')), chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content="What's up how can i guide you", additional_kwargs={}, response_metadata={})]), buffer='New summary: \n\nThe human greeted the AI with "hi". The AI responded asking what the human needed help with. \n', memory_key='summary')

In [34]:
memory = ConversationSummaryMemory.from_messages(
    llm=model,
    chat_memory=history,
    memory_key="summary",
    human_prefix="User",
    ai_prefix="AI"
)


In [35]:
memory.buffer

'User greets the AI with a simple "hi". The AI responds by asking how it can be of assistance. \n'

#### Building chain with model and the conversation summary memory

In [36]:
from langchain.chains import ConversationChain

In [37]:
conversation_with_summary = ConversationChain(
    llm=model,
    memory=ConversationSummaryMemory(llm=model),
    verbose=True
)

In [38]:
conversation_with_summary.predict(input="Hi, What's up")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, What's up
AI:

> Finished chain.


"Hi there! Not much, just hanging out in the digital world, waiting to be helpful. What about you? What's going on in your world today?  \n\n"

In [39]:
conversation_with_summary.predict(input="Can you explain MLOPS pipiline in 6 points")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
New summary:  A human greets the AI and asks what it's up to. The AI responds by saying it's waiting to be helpful and asks the human what's going on in their world. 

Human: Can you explain MLOPS pipiline in 6 points
AI:

> Finished chain.


'Of course!  The MLOps pipeline is a crucial part of getting machine learning models out into the real world. Here\'s a breakdown in 6 points:\n\n1. **Data Collection and Preparation:** This is the foundation!  It involves gathering relevant data, cleaning it, transforming it into a usable format, and splitting it into training, validation, and testing sets.\n\n2. **Model Training:** This is where the "learning" happens.  Algorithms are used to train the model on the prepared data, adjusting parameters to improve its performance.  \n\n3. **Model Evaluation:**  We need to see how well the model is doing!  This involves using the validation set to assess metrics like accuracy, precision, recall, and F1-score.\n\n4. **Model Deployment:**  Once the model is deemed good enough, it\'s deployed to a production environment where it can be used to make predictions on new data. This could be a web server, a cloud platform, or even embedded in a device.\n\n5. **Model Monitoring:**  We can\'t just

In [40]:
conversation_with_summary.predict(input="what was the third point")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
A human greets the AI and asks what it's up to. The AI responds by saying it's waiting to be helpful and asks the human what's going on in their world. The human then asks the AI to explain the MLOps pipeline in 6 points. The AI obliges, outlining the steps involved: data collection and preparation, model training, model evaluation, model deployment, model monitoring, and model retraining.  

Human: what was the third point
AI:

> Finished chain.


"The third point was **model evaluation**.  \n\nIs there anything else you'd like to know about the MLOps pipeline? I'm happy to elaborate on any of the steps! \n\n"

In [41]:
conversation_with_summary.memory.buffer

"A human greets the AI and asks what it's up to. The AI responds by saying it's waiting to be helpful and asks the human what's going on in their world. The human then asks the AI to explain the MLOps pipeline in 6 points. The AI obliges, outlining the steps: data collection and preparation, model training, model evaluation, model deployment, model monitoring, and model retraining. The human then asks for clarification on the third point, which the AI identifies as model evaluation.  The AI offers to elaborate on any of the steps further. \n"

#### **Conversation Summary Buffer Memory**

While summary is good, we know that recent conversation has high correlation to upcoming query and

A summary of old conversation with a buffer memory of last few conversation would be a good combination. This class exactly does that.

You can set the token limit which define how much historical conversation to be kept along with the summary. Higher the token size more the exact conversation history kept as-it-is.

In [42]:
from langchain.memory import ConversationSummaryBufferMemory

##### Same as above going with type-1

In [43]:
memory2 = ConversationSummaryBufferMemory(
    llm=model,
    return_message=True
)


In [44]:
memory2.save_context(
    {"input":"My name is Yogi I love working with AIDS"},
    {"Output":"That's nice to meet you, How are you?"}

)

memory2.save_context(
    {"input":"Yeah currently i am exploring Agentic AI"},
    {"output":"Vry godd its very nice technology"}
)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [45]:
memory2.load_memory_variables({})

{'history': "Human: My name is Yogi I love working with AIDS\nAI: That's nice to meet you, How are you?\nHuman: Yeah currently i am exploring Agentic AI\nAI: Vry godd its very nice technology"}

In [49]:
moemory2 = ConversationSummaryBufferMemory(llm=model,return_message=True,max_token_limit=50)

In [51]:
memory2.save_context(
    {"input": "Sunny and Mayank are working on a hackathon project."},
    {"output": "That's awesome! What's the hackathon project about?"}
)
memory2.save_context(
    {"input": "It's a machine learning project focused on healthcare."},
    {"output": "Sounds exciting! Are they building a prediction model or something else?"}
)
memory2.save_context(
    {"input": "Yes, they’re working on a model to predict patient outcomes."},
    {"output": "Impressive! Wishing Sunny and Mayank all the best for their project."}
)

In [52]:
memory2.load_memory_variables({})["history"]

"Human: My name is Yogi I love working with AIDS\nAI: That's nice to meet you, How are you?\nHuman: Yeah currently i am exploring Agentic AI\nAI: Vry godd its very nice technology\nHuman: Sunny and Mayank are working on a hackathon project.\nAI: That's awesome! What's the hackathon project about?\nHuman: Sunny and Mayank are working on a hackathon project.\nAI: That's awesome! What's the hackathon project about?\nHuman: It's a machine learning project focused on healthcare.\nAI: Sounds exciting! Are they building a prediction model or something else?\nHuman: Yes, they’re working on a model to predict patient outcomes.\nAI: Impressive! Wishing Sunny and Mayank all the best for their project."

In [53]:
# Making chain
from langchain import ConversationChain

conversation_with_summary_buffer = ConversationChain(
    llm=model,
    memory=ConversationSummaryBufferMemory(llm=model,max_token_limit=50),
    verbose=True
)

In [54]:
conversation_with_summary_buffer.predict(input="How are you ?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: How are you ?
AI:

> Finished chain.


"AI: As an AI, I don't have feelings, but I'm here and ready to assist you!  What can I do for you today? 😊  \n\n"

In [55]:
conversation_with_summary_buffer.predict(input="If you dont have feelings i dont wanted to speak to you")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: How are you ?
AI: AI: As an AI, I don't have feelings, but I'm here and ready to assist you!  What can I do for you today? 😊  


Human: If you dont have feelings i dont wanted to speak to you
AI:

> Finished chain.


"That's understandable! It's important to connect with others on a human level.  If you'd rather speak to someone who can share feelings, I completely understand.  \n\nWould you like me to help you find some resources for connecting with people? Perhaps a forum or online community that aligns with your interests?  \n\n"

In [58]:
conversation_with_summary_buffer.memory.buffer

"System: The human asks the AI how it is. The AI, stating it doesn't have feelings, offers to assist. The human says they don't want to speak to it because it doesn't have feelings. The AI understands and offers to help the human find resources for connecting with people. \n"